In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, datasets, transforms

In [2]:
import sys
print(sys.executable)

/home/iztihad/venvs/ml/bin/python


In [3]:
model_config = {
    "batch_size": 16,
    "input_size": 224,
    "architecture": "effiecientnet_v2",
    "learning_rate": 0.001,
    "epochs": 20,
    "pretrained":True
}

In [4]:
data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.Grayscale(num_output_channels=3),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],
                             [0.229,0.224,0.225])
    ]),

    'val': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.Grayscale(num_output_channels=3),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],
                             [0.229,0.224,0.225])
    ]),

    'test': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.Grayscale(num_output_channels=3),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],
                             [0.229,0.224,0.225])
    ])
}

train_dir = "../BanglaLekha_dataset_split/train"
val_dir = "../BanglaLekha_dataset_split/validation"
test_dir = "../BanglaLekha_dataset_split/test"


train_dataset = datasets.ImageFolder(root=train_dir, transform=data_transforms["train"])
val_dataset = datasets.ImageFolder(root=val_dir, transform=data_transforms["val"])
test_dataset = datasets.ImageFolder(root=test_dir, transform=data_transforms["test"])

train_dataloader = DataLoader(train_dataset, batch_size=model_config["batch_size"], shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=model_config["batch_size"], shuffle=False)
test_dataloader = DataLoader(test_dataset, batch_size=model_config["batch_size"], shuffle=False)

In [5]:

efficientnet_b3 = models.efficientnet_b3(pretrained=True)
efficientnet_b3 = models.efficientnet_b3(pretrained=True)

for param in efficientnet_b3.parameters():
    param.requires_grad = False

in_features = efficientnet_b3.classifier[1].in_features
efficientnet_b3.classifier[1] = nn.Linear(in_features, 84)

total_params = sum(p.numel() for p in efficientnet_b3.parameters())

gpu = torch.device("cuda")
efficientnet_b3 = efficientnet_b3.to(gpu)


/home/iztihad/venvs/ml/lib/python3.12/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/iztihad/venvs/ml/lib/python3.12/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=EfficientNet_B3_Weights.IMAGENET1K_V1`. You can also use `weights=EfficientNet_B3_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [6]:
print(total_params)

10825340


In [7]:
import fine_tuning as ft
ft.fine_tune(model=efficientnet_b3, model_name="efficientnet_b3", state="full") #Change the state for fine-tuning

EfficientNet(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 40, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(40, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): SiLU(inplace=True)
    )
    (1): Sequential(
      (0): MBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(40, 40, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=40, bias=False)
            (1): BatchNorm2d(40, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (2): SiLU(inplace=True)
          )
          (1): SqueezeExcitation(
            (avgpool): AdaptiveAvgPool2d(output_size=1)
            (fc1): Conv2d(40, 10, kernel_size=(1, 1), stride=(1, 1))
            (fc2): Conv2d(10, 40, kernel_size=(1, 1), stride=(1, 1))
            (activation): SiLU(inplace=True)
            (scale_activation): Sigmoid()
          )
          (2): Conv2dNormActiv

In [8]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW([
    {"params": efficientnet_b3.classifier[1].parameters(), "lr": 1e-4, "weight-decay": 1e-4},
    {"params": efficientnet_b3.features.parameters(), "lr": 1e-5, "weight-decay": 1e-4},
    
])
epochs = model_config["epochs"]

In [9]:
def validate_model(model, val_dataloader):
    with torch.no_grad():
        model.eval()
        total = 0
        total_correct = 0

        for images, labels in val_dataloader:
            images = images.to(gpu)
            labels = labels.to(gpu)

            output = model(images)
            _, predicted = torch.max(output, 1)

            total = total + len(labels)
            total_correct = total_correct + (predicted == labels).sum().item()

        return total_correct/total 



In [10]:
def train_model(model, train_dataloader, val_dataloader, optimizer, criterion, epochs):
    
    max_val_accuracy = 0
    count = 0
    patience = 5

    for epoch in range(epochs):
        model.train()
        total_loss = 0

        for images, label in train_dataloader:
            images = images.to(gpu)
            label = label.to(gpu)

            optimizer.zero_grad()
            output = model(images)
            loss = criterion(output, label)
            loss.backward()
            optimizer.step()

            total_loss = total_loss + loss.item()
        
        val_accuracy = validate_model(efficientnet_b3, val_dataloader)

        if(val_accuracy > max_val_accuracy):
            max_val_accuracy = val_accuracy
            count = 0

            torch.save(model.state_dict(), "saved_parameters/efficientnet_b3.pth")

        else:
            count = count + 1

        if(count >= patience):
            break

        

        print(f"Epoch: {epoch + 1}, Training Loss: {total_loss/len(train_dataloader)}, Validation Accuracy: {val_accuracy}")

In [11]:
train_model(efficientnet_b3, train_dataloader, val_dataloader, optimizer, criterion, model_config["epochs"])

Epoch: 1, Training Loss: 1.3433509601024496, Validation Accuracy: 0.9029495144459859
Epoch: 2, Training Loss: 0.3896035482131008, Validation Accuracy: 0.9294287954641414
Epoch: 3, Training Loss: 0.29205844913197615, Validation Accuracy: 0.9386573375957536
Epoch: 4, Training Loss: 0.2428770152049627, Validation Accuracy: 0.9436636709089813
Epoch: 5, Training Loss: 0.21124302403986941, Validation Accuracy: 0.9461969962000121
Epoch: 6, Training Loss: 0.18589054326747087, Validation Accuracy: 0.9474636588455275
Epoch: 7, Training Loss: 0.16651743249921142, Validation Accuracy: 0.9490319078352132
Epoch: 8, Training Loss: 0.14794277200698122, Validation Accuracy: 0.9499366668677243
Epoch: 9, Training Loss: 0.13320024805500916, Validation Accuracy: 0.9494541287170517
Epoch: 10, Training Loss: 0.11952895431030995, Validation Accuracy: 0.9488509560287111
Epoch: 11, Training Loss: 0.10772023667841552, Validation Accuracy: 0.9493334941793835
Epoch: 12, Training Loss: 0.09802870959734565, Validati

In [12]:
efficientnet_b3.load_state_dict(torch.load("saved_parameters/efficientnet_b3.pth"))

<All keys matched successfully>

In [13]:
accuracy = validate_model(efficientnet_b3, test_dataloader)
print(f"Accuracy: {100 * accuracy}")

Accuracy: 94.6048244164739
